O código foi executado em ambiente COLAB

In [110]:
# Importar bibliotecas necessárias

# Deep Learning / Machine Learning
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# EDA
import pandas as pd
import plotly.express as px

### Carga dos Dados

In [111]:
# Carregar o dataset
df_veiculos = pd.read_csv('data/veiculos.csv')

### Exploração Inicial dos Dados

In [112]:
### Estrutura do dataset
df_veiculos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Categoria                        835 non-null    int64  
 1   Cor                              835 non-null    object 
 2   Pais de Origem                   835 non-null    object 
 3   Ano Modelo                       835 non-null    int64  
 4   Ano Fabricação                   835 non-null    int64  
 5   Potencia                         835 non-null    int64  
 6   Quantidade de lugares            835 non-null    int64  
 7   Unico dono?                      835 non-null    int64  
 8   Ja teve sinistro?                835 non-null    int64  
 9   Ja foi carro de aplicativo?      835 non-null    int64  
 10  Revisoes em dia?                 835 non-null    int64  
 11  Sistema avancado de Multimidia?  835 non-null    int64  
 12  Tipo de Motorizacao   

In [113]:
# Visualizar as primeiras linhas do dataset
df_veiculos.head()

,Categoria,Cor,Pais de Origem,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Tipo de Motorizacao,Kilometragem,Tipo de Transmissao,Tamanho do porta malas,Valor de Venda
0,4,azul,Alemanha,2026,2025,143,4,0,0,1,1,1,Híbrido,60459,3,430,112898.39
1,7,verde,Alemanha,2024,2024,541,5,0,0,1,1,0,Híbrido,105982,7,484,887822.26
2,2,prata,Japão,2026,2025,94,4,0,1,1,0,0,Flex,38626,3,321,55516.43
3,4,preto,Japão,2022,2021,159,4,1,1,0,0,1,Flex,91185,2,415,147030.87
4,2,azul,Coreia do Sul,2026,2025,114,4,0,0,0,0,1,Flex,26037,3,381,93719.67


In [114]:
# Visualizar as ultimas linhas do dataset
df_veiculos.tail()

,Categoria,Cor,Pais de Origem,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Tipo de Motorizacao,Kilometragem,Tipo de Transmissao,Tamanho do porta malas,Valor de Venda
830,5,prata,Coreia do Sul,2026,2025,244,5,1,0,1,0,0,Híbrido,43509,2,522,214388.05
831,6,prata,Japão,2022,2021,258,5,0,0,1,0,1,Flex,210838,6,464,354138.90
832,5,vermelho,Coreia do Sul,2023,2022,201,5,1,1,1,1,1,Híbrido,272508,3,471,106276.30
833,5,verde,Estados Unidos,2024,2023,241,5,0,0,0,1,0,Elétrico,45229,4,471,213261.36
834,8,branco,Inglaterra,2022,2022,1239,2,0,0,0,0,1,Elétrico,89190,9,197,8079545.72


In [115]:
# Estatísticas descritivas do dataset
df_veiculos.describe()

,Categoria,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Kilometragem,Tipo de Transmissao,Tamanho do porta malas,Valor de Venda
count,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,8.350000e+02
mean,4.453892,2024.037126,2023.428743,307.476647,4.426347,0.492216,0.534132,0.462275,0.489820,0.489820,96439.767665,3.785629,395.049102,7.902733e+05
std,2.274606,1.402229,1.358211,320.806721,1.303114,0.500239,0.499133,0.498874,0.500196,0.500196,78575.237811,2.169836,131.769333,1.742400e+06
min,1.000000,2022.000000,2021.000000,70.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12611.000000,1.000000,80.000000,4.280000e+04
25%,2.000000,2023.000000,2022.000000,114.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,42289.500000,2.000000,308.000000,8.187166e+04
50%,4.000000,2024.000000,2024.000000,160.000000,4.000000,0.000000,1.000000,0.000000,0.000000,0.000000,70509.000000,3.000000,428.000000,1.436343e+05
75%,6.000000,2025.000000,2025.000000,371.500000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,129393.500000,5.000000,490.500000,4.210006e+05
max,8.000000,2026.000000,2025.000000,1485.000000,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000,403947.000000,9.000000,600.000000,9.433305e+06


In [116]:
# Mostrar valores únicos de cada coluna
for column in df_veiculos.select_dtypes(include=['object']).columns:
    unique_values = df_veiculos[column].unique()
    print(f'Coluna: {column}, Valores únicos: {unique_values}')

Coluna: Cor, Valores únicos: ['azul' 'verde' 'prata' 'preto' 'cinza' 'vermelho' 'branco']
Coluna: Pais de Origem, Valores únicos: ['Alemanha' 'Japão' 'Coreia do Sul' 'Inglaterra' 'França' 'Itália'
 'Estados Unidos' 'China']
Coluna: Tipo de Motorizacao, Valores únicos: ['Híbrido' 'Flex' 'Elétrico' 'Gasolina']


### Preparação de Dados para EDA

In [117]:
# Criar lista de variáveis categóricas
categorical_features = df_veiculos.select_dtypes(include=['object']).columns.tolist()
# Incluir variáveis categóricas adicionais
additional_categorical_features = ['Categoria', 'Ano Modelo', 'Ano Fabricação', \
                            'Unico dono?', 'Ja teve sinistro?', \
                            'Ja foi carro de aplicativo?', \
                            'Revisoes em dia?', \
                            'Sistema avancado de Multimidia?', \
                            'Tipo de Transmissao']
categorical_features.extend(additional_categorical_features)
categorical_features

['Cor',
 'Pais de Origem',
 'Tipo de Motorizacao',
 'Categoria',
 'Ano Modelo',
 'Ano Fabricação',
 'Unico dono?',
 'Ja teve sinistro?',
 'Ja foi carro de aplicativo?',
 'Revisoes em dia?',
 'Sistema avancado de Multimidia?',
 'Tipo de Transmissao']

In [118]:
# Criar lista de variáveis numéricas
numerical_features = df_veiculos.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remover variável 'Valor de Venda' da lista de variáveis numéricas
numerical_features.remove('Valor de Venda')
# Remover variáveis categóricas numéricas da lista de variáveis numéricas
numerical_features = [feature for feature in numerical_features if feature not in additional_categorical_features]
numerical_features

['Potencia', 'Quantidade de lugares', 'Kilometragem', 'Tamanho do porta malas']

In [119]:
# Variável alvo
target = ['Valor de Venda']

### EDA - Análise Exploratória de Dados

In [120]:
# Distribuição da variável alvo, usando plotly
fig = px.histogram(df_veiculos, x='Valor de Venda', nbins=50, title='Distribuição do Valor de Venda')
fig.show()

In [121]:
# Distribuição das variáveis numéricas
for feature in numerical_features:
    fig = px.histogram(df_veiculos, x=feature, nbins=50, title=f'Distribuição de {feature}')
    fig.show()

In [122]:
# Distribuição das variáveis categóricas
for feature in categorical_features:
    fig = px.histogram(df_veiculos, x=feature, title=f'Distribuição de {feature}')
    fig.show()

In [123]:
# Boxplot das variáveis numéricas
for feature in numerical_features:
    fig = px.box(df_veiculos, y=feature, title=f'Boxplot de {feature}')
    fig.show()

In [124]:
# Boxplot das variáveis categóricas com target
for feature in categorical_features:
    fig = px.box(df_veiculos, x=feature, y='Valor de Venda', title=f'Boxplot de Valor de Venda por {feature}')
    fig.show()

### Preparar dados para Correlações

In [125]:
# Transformar variáveis categóricas em numéricas usando one-hot encoding
df_veiculos_encoded = pd.get_dummies(df_veiculos, dtype=int)
df_veiculos_encoded.head()

,Categoria,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,...,Pais de Origem_Coreia do Sul,Pais de Origem_Estados Unidos,Pais de Origem_França,Pais de Origem_Inglaterra,Pais de Origem_Itália,Pais de Origem_Japão,Tipo de Motorizacao_Elétrico,Tipo de Motorizacao_Flex,Tipo de Motorizacao_Gasolina,Tipo de Motorizacao_Híbrido
0,4,2026,2025,143,4,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,1
1,7,2024,2024,541,5,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,1
2,2,2026,2025,94,4,0,1,1,0,0,...,0,0,0,0,0,1,0,1,0,0
3,4,2022,2021,159,4,1,1,0,0,1,...,0,0,0,0,0,1,0,1,0,0
4,2,2026,2025,114,4,0,0,0,0,1,...,1,0,0,0,0,0,0,1,0,0


In [126]:
# Estrutura do dataset após one-hot encoding
df_veiculos_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Categoria                        835 non-null    int64  
 1   Ano Modelo                       835 non-null    int64  
 2   Ano Fabricação                   835 non-null    int64  
 3   Potencia                         835 non-null    int64  
 4   Quantidade de lugares            835 non-null    int64  
 5   Unico dono?                      835 non-null    int64  
 6   Ja teve sinistro?                835 non-null    int64  
 7   Ja foi carro de aplicativo?      835 non-null    int64  
 8   Revisoes em dia?                 835 non-null    int64  
 9   Sistema avancado de Multimidia?  835 non-null    int64  
 10  Kilometragem                     835 non-null    int64  
 11  Tipo de Transmissao              835 non-null    int64  
 12  Tamanho do porta malas

In [127]:
# Mostrar a correlação entre as variáveis numéricas usando plotly
fig = px.imshow(df_veiculos_encoded.corr(), text_auto=True, aspect="auto", title='Matriz de Correlação', width=1080, height=900)
fig.show()

In [128]:
# Mostrar o Top 5 de correlações com a variável alvo 'Valor de Venda'
correlation_target = df_veiculos_encoded.corr()['Valor de Venda'].sort_values(ascending=False)
print("Top 5 correlações com 'Valor de Venda':")
print(correlation_target.head(5))

Top 5 correlações com 'Valor de Venda':
Valor de Venda                  1.000000
Potencia                        0.827074
Tipo de Transmissao             0.648501
Categoria                       0.587998
Tipo de Motorizacao_Gasolina    0.347504
Name: Valor de Venda, dtype: float64


In [129]:
# Transformar variáveis categóricas em numéricas usando one-hot encoding
df_veiculos_encoded = pd.get_dummies(df_veiculos, columns=categorical_features, dtype=int)
df_veiculos_encoded.head()

,Potencia,Quantidade de lugares,Kilometragem,Tamanho do porta malas,Valor de Venda,Cor_azul,Cor_branco,Cor_cinza,Cor_prata,Cor_preto,...,Sistema avancado de Multimidia?_1,Tipo de Transmissao_1,Tipo de Transmissao_2,Tipo de Transmissao_3,Tipo de Transmissao_4,Tipo de Transmissao_5,Tipo de Transmissao_6,Tipo de Transmissao_7,Tipo de Transmissao_8,Tipo de Transmissao_9
0,143,4,60459,430,112898.39,1,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0
1,541,5,105982,484,887822.26,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,94,4,38626,321,55516.43,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
3,159,4,91185,415,147030.87,0,0,0,0,1,...,1,0,1,0,0,0,0,0,0,0
4,114,4,26037,381,93719.67,1,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0


In [130]:
# Estrutura do dataset após one-hot encoding
df_veiculos_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 61 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Potencia                           835 non-null    int64  
 1   Quantidade de lugares              835 non-null    int64  
 2   Kilometragem                       835 non-null    int64  
 3   Tamanho do porta malas             835 non-null    int64  
 4   Valor de Venda                     835 non-null    float64
 5   Cor_azul                           835 non-null    int64  
 6   Cor_branco                         835 non-null    int64  
 7   Cor_cinza                          835 non-null    int64  
 8   Cor_prata                          835 non-null    int64  
 9   Cor_preto                          835 non-null    int64  
 10  Cor_verde                          835 non-null    int64  
 11  Cor_vermelho                       835 non-null    int64  

In [131]:
# Mostrar a correlação entre as variáveis numéricas usando plotly
fig = px.imshow(df_veiculos_encoded.corr(), text_auto=True, aspect="auto", title='Matriz de Correlação', width=1080, height=900)
fig.show()

In [132]:
# Mostrar o Top 5 de correlações com a variável alvo 'Valor de Venda'
correlation_target = df_veiculos_encoded.corr()['Valor de Venda'].sort_values(ascending=False)
print("Top 5 correlações com 'Valor de Venda':")
print(correlation_target.head(5))

Top 5 correlações com 'Valor de Venda':
Valor de Venda           1.000000
Categoria_8              0.899460
Potencia                 0.827074
Tipo de Transmissao_8    0.508115
Tipo de Transmissao_9    0.500085
Name: Valor de Venda, dtype: float64


### Preparar dados para treinamento da rede neural

In [133]:
# Dividir o dataset em features (X) e target (y)
X = df_veiculos.drop(columns=target, axis=1)
y = np.array(df_veiculos[target])

In [134]:
# Separar entre Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42, shuffle=True)

In [135]:
# Dividir Teste entre Validação e Teste final
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42, shuffle=True)

In [136]:
# Criar Pipeline de pré-processamento dos dados
reduced_categorical_features = ['Categoria', 'Ano Modelo', 'Ano Fabricação', 'Cor', \
                                'Tipo de Transmissao', 'Pais de Origem', 'Tipo de Motorizacao']

# Aplicar Transformações por tipo
numeric_transformer = MinMaxScaler()
categorical_transformer = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, reduced_categorical_features),
    ]
)

In [137]:
# Aplicar PreProcessor para normalizar os dados
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

scaler_Y = MinMaxScaler()
y_train = scaler_Y.fit_transform(y_train.reshape(-1, 1))
y_val = scaler_Y.transform(y_val.reshape(-1, 1))
y_test = scaler_Y.transform(y_test.reshape(-1, 1))

In [138]:
# Salvar Scaler para uso futuro
import joblib
joblib.dump(preprocessor, 'model/preprocessor_keras.pkl')

['model/preprocessor_keras.pkl']

In [139]:
# Visualizar Shape dos datasets
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (417, 50), y_train: (417, 1)
X_val: (209, 50), y_val: (209, 1)
X_test: (209, 50), y_test: (209, 1)


In [140]:
# Define batch size
BATCH_SIZE = 32

# Create tf.data.Dataset objects
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(BATCH_SIZE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

### Criar Arquitetura de Rede

In [141]:
# Criar uma arquitetura de rede neural com 3 camadas ocultas, usando ReLU como função de ativação
def create_model(input_size, hidden_layer_sizes=[128, 64, 32, 16], output_size=1, dropout_rate=0.2):
    model = models.Sequential([
        # Input layer
        layers.Input(shape=(input_size,)),

        # First hidden layer
        layers.Dense(hidden_layer_sizes[0]),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(dropout_rate),

        # Second hidden layer
        layers.Dense(hidden_layer_sizes[1]),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(dropout_rate),

        # Third hidden layer
        layers.Dense(hidden_layer_sizes[2]),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(dropout_rate),

        # Fourth hidden layer
        layers.Dense(hidden_layer_sizes[3]),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(dropout_rate),

        # Output layer
        layers.Dense(output_size)
    ])

    return model

In [142]:
# Instanciar o modelo
model = create_model(input_size=X_train.shape[1], hidden_layer_sizes=[64, 32, 16, 8], output_size=1, dropout_rate=0.2)

In [143]:
# Compilar e visualizar o modelo
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mse']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 64)             │         3,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_8 (ReLU)                  │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_9 (ReLU)                  │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_10 (ReLU)                 │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 8)              │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_11 (ReLU)                 │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,497 (25.38 KB)

 Trainable params: 6,257 (24.44 KB)

 Non-trainable params: 240 (960.00 B)

### Treinar a rede neural

In [144]:
# Define callbacks para Early Stopping
patience = 35
min_delta = 1e-6

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=patience,
    min_delta=min_delta,
    restore_best_weights=True
)

NUM_EPOCHS = 100

# Treinamento do modelo
history = model.fit(
    train_dataset,
    epochs=NUM_EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - loss: 0.7292 - mse: 0.7292 - val_loss: 0.0464 - val_mse: 0.0464
Epoch 2/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6881 - mse: 0.6881 - val_loss: 0.0421 - val_mse: 0.0421
Epoch 3/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4665 - mse: 0.4665 - val_loss: 0.0392 - val_mse: 0.0392
Epoch 4/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4526 - mse: 0.4526 - val_loss: 0.0372 - val_mse: 0.0372
Epoch 5/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3781 - mse: 0.3781 - val_loss: 0.0364 - val_mse: 0.0364
Epoch 6/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4301 - mse: 0.4301 - val_loss: 0.0363 - val_mse: 0.0363
Epoch 7/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4739 - mse: 0.4739 - val_loss: 0.0364 - val_mse: 0.0364
Epoch 8/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2495 - mse: 0.2495 - val_loss: 0.0368 - val_mse: 0.0368
Epoch 9/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.36

### Visualizar Resultados do Treinamento

In [145]:
# Plotar as perdas de treino e validação usando plotly
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history['loss'], mode='lines', name='Loss - Treino'))
fig.add_trace(go.Scatter(y=history.history['val_loss'], mode='lines', name='Loss - Val'))
fig.update_layout(title='Loss - Treino e Validação', xaxis_title='Época', yaxis_title='Loss')
fig.show()

In [146]:
history.history['loss'][-1], history.history['val_loss'][-1]

(0.022335100919008255, 0.009246169589459896)

### Validar as métricas no conjunto de teste

In [147]:
# Validar as métricas no conjunto de teste
test_results = model.evaluate(test_dataset, verbose=1)
print(f"Test loss: {test_results[0]:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0107 - mse: 0.0107
Test loss: 0.0089


In [148]:
history.history['loss'][-1], history.history['val_loss'][-1], test_results[0]

(0.022335100919008255, 0.009246169589459896, 0.00887923501431942)

### Salvar Modelo

In [149]:
model.save('model/best_model.keras')